# Апта 16 — RNN / LSTM / GRU үй тапсырмасы
Бұл ноутбук теориялық және практикалық тапсырмаларды қамтиды.

## 1️⃣ Теориялық тапсырмалар

### 1.1 RNN дегеніміз не? Feedforward-тан айырмашылығы
**RNN (Recurrent Neural Network)** — тізбекті (sequence) деректермен жұмыс істейтін нейрондық желі.  
Негізгі ерекшелігі — алдыңғы қадамнан келген ақпаратты **hidden state** арқылы сақтайды.

**Айырмашылығы (қысқаша):**
- **Feedforward:** әр input тәуелсіз, жады (memory) жоқ  
- **RNN:** уақытқа/ретке тәуелді, hidden state арқылы контекст сақтайды

---

### 1.2 RNN қандай есептерде тиімді қолданылады? (кемінде 4 мысал)
1) Мәтін өңдеу (next-word prediction, translation)  
2) Уақыттық қатарлар (температура, акция бағасы, сенсор деректері)  
3) Speech recognition (дыбысты мәтінге айналдыру)  
4) Диалог/чат-боттар  
(Қосымша: биологиялық тізбектер — DNA/ақуыз)

---

### 1.3 Hidden state ұғымы
**Hidden state (hₜ)** — алдыңғы қадамдардан жиналған контекст/ақпарат.  
Жалпы түрі:
\[
h_t = f(W_h h_{t-1} + W_x x_t)
\]
Мақсаты: модельге «есте сақтау» мүмкіндігін беру.

---

### 1.4 Vanishing gradient және exploding gradient
- **Vanishing gradient:** градиент нөлге жақындайды → ерте қабаттар/ерте уақыт қадамдары дұрыс үйренбейді.  
- **Exploding gradient:** градиент өте үлкен болып кетеді → оқыту тұрақсызданады (loss “секіреді”).

---

### 1.5 LSTM және GRU не үшін ұсынылды? RNN-нен айырмасы
**Мақсаты:** ұзын тәуелділіктерді жақсы сақтау және vanishing gradient әсерін азайту.

- **LSTM:** input/forget/output gate + cell state (ұзақ жады)  
- **GRU:** gate-тері ықшам (update/reset), параметрі аздау

---

### 1.6 RNN-ді қай кезде қолдану тиімсіз?
- Sequence өте ұзын болғанда және параллельдеу қажет болғанда (Transformer жиі тиімді)  
- Уақытқа тәуелділік жоқ деректерде (feedforward/CNN жеткілікті)  
- Өте үлкен датасетте жылдамдық/параллель есептеу критик болса


## 2️⃣ Терең теория

### 2.3 Неліктен ұзын тізбектерде градиент нөлге жақындайды? (туынды, tanh/sigmoid)
BPTT кезінде градиент көбінесе мынаған ұқсас көбейтінді арқылы таралады:
\[
\frac{\partial L}{\partial h_{t-k}} = \frac{\partial L}{\partial h_t} \prod_{i=t-k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}}
\]

Егер активтендіру **sigmoid** болса:
- \(\sigma'(z)=\sigma(z)(1-\sigma(z))\), ең үлкен мәні ≈ **0.25**.

Егер активтендіру **tanh** болса:
- \(\tanh'(z)=1-\tanh^2(z)\), ол да көбіне **< 1**.

Осылайша, көп қадамда \(<1\) сандардың көбейтіндісі **нөлге жақындайды** → **vanishing gradient**.

---

### 2.4 Backpropagation Through Time (BPTT) деген не? Неге RNN оқытуы қиынырақ?
**BPTT** — RNN-ді уақыт бойынша “ашып” (unroll) алып, сол ашылған желіге backpropagation қолдану.

Неге қиынырақ:
- Уақыт қадамдары көп → есептеу/жад көп қажет  
- Ұзын тәуелділіктер → vanishing/exploding gradient ықтималдығы жоғары  
- Оқыту баяу, тұрақтандыру үшін қосымша трюктар керек (gradient clipping, LSTM/GRU, good init т.б.)


## 3️⃣ Практикалық тапсырмалар (ойлануға)

### 3.1 [2, 4, 6, 8, 10] қатарынан келесі мәнді RNN логикасымен болжау
RNN әр қадамда мәнді қабылдап, hidden state-ке “үрдісті” жинайды.  
Бұл қатарда заңдылық: **әр жолы +2**.  
Оқығаннан кейін модель келесі мәнді \(\approx 12\) деп болжауы тиіс.

---

### 3.2 Мәтіндік деректер үшін RNN
**Input қалай беріледі?**
- Әр сөз/таңба → индекс (token id)  
- Кейін one-hot немесе embedding арқылы векторға айналады

**One-hot encoding не үшін керек?**
- Категориялық сөздерді сандық форматқа түсіреді  
- Модельге “қай сөз келгенін” айқын көрсетеді (бірақ өлшемі үлкен болуы мүмкін)

---

### 3.3 LSTM қажет болатын нақты 3 жағдай
1) Ұзын мәтіндегі мағыналық байланыстар (мыс: реферат/мақала)  
2) Маусымдық уақыттық қатарлар (климат, энергия тұтыну)  
3) Speech/аудио тізбегі (ұзын контекст керек)


In [ ]:
# 4️⃣ Keras арқылы бір қабатты RNN моделі
# Тoy мысал: [2,4,6,8] -> 10 сияқты үлгі арқылы үйрету


In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam

# Бір ғана үлгі: [2,4,6,8] -> 10
X = np.array([[2, 4, 6, 8]], dtype=np.float32).reshape((1, 4, 1))
y = np.array([10], dtype=np.float32)

model = Sequential([
    SimpleRNN(16, activation="tanh", input_shape=(4, 1)),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.01), loss="mse")
model.summary()

history = model.fit(X, y, epochs=300, verbose=0)

pred = model.predict(X, verbose=0)
pred


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 16)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305 (1.19 KB)

 Trainable params: 305 (1.19 KB)

 Non-trainable params: 0 (0.00 B)

array([[9.999999]], dtype=float32)

In [ ]:
# Егер қатар [2,4,6,8,10] болса, келесі мән ≈ 12 деп күтеміз.
# Мұнда дәл болжау үшін көп үлгі керек, ал біз тек "toy" түрде көрсеттік.

## 5️⃣ Қосымша (жобаға арналған идеялар)

- 📈 **Ауа температурасының уақыттық қатарын болжау** (LSTM)  
- 📝 **Қазақ тіліндегі мәтінге символдық RNN** (келесі әріпті болжау)  
- 💬 **Чат-хабарламалардың келесі сөзін болжау** (word-level RNN/LSTM)  
- 💰 **Валюта бағамын болжау** (toy example, sliding window)
